# 🇸🇦 Arabic Piper TTS Fine-Tuning — Complete Pipeline

Run each section top-to-bottom. On disconnect, re-run Section 1 — training auto-resumes.

### Sections:
1. ⚙️ Environment Setup
2. 📦 Dataset Download & Preparation
3. 🔊 Baseline Benchmark
4. 🏋️ Fine-Tuning
5. 📊 Export & Comparison

---
## ⚙️ Section 1: Environment Setup

In [ ]:
# 1.1 Clone Repository
import os
REPO_URL = 'https://github.com/YOUR_USERNAME/piper-tts-finetuning.git'  # <-- EDIT
REPO_DIR = '/content/piper-tts-finetuning'
if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# 1.2 GPU Check
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# 1.3 Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1.4 Set up Drive Directories
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/Arabic-Piper')
for s in ['datasets', 'processed', 'checkpoints', 'tensorboard', 'logs', 'outputs', 'metrics']:
    (DRIVE_ROOT / s).mkdir(parents=True, exist_ok=True)
    print(f'Verified: {DRIVE_ROOT / s}')

In [ ]:
# 1.5 Install System & Python Dependencies
import sys; print(f'Python: {sys.version}')
!apt-get update -q && apt-get install -y -q espeak-ng libespeak-ng-dev build-essential
!pip install -q -r requirements.txt
print('\n✅ Core dependencies installed.')

In [ ]:
# 1.6 Install Piper Training Engine, Apply Compatibility Patches & Compile monotonic_align
import os, subprocess, sys, sysconfig
from pathlib import Path

PIPER_SRC = '/content/piper'
if not os.path.exists(PIPER_SRC):
    os.system('git clone https://github.com/rhasspy/piper.git /content/piper')

# pytorch-lightning 1.9.5 required
os.system('pip install -q cython setuptools "pytorch-lightning~=1.9.5"')
os.system(f'pip install -q --no-deps -e {PIPER_SRC}/src/python')

# Patch 1: PyTorch 2.6 weights_only checkpoint unpickling
main_py = Path(f'{PIPER_SRC}/src/python/piper_train/__main__.py')
if main_py.exists():
    content = main_py.read_text()
    patch = 'import pathlib, torch\ntry:\n    torch.serialization.add_safe_globals([pathlib.PosixPath, pathlib.WindowsPath])\nexcept Exception:\n    pass\n\n'
    if 'add_safe_globals' not in content:
        main_py.write_text(patch + content)
        print('✅ Patched piper_train/__main__.py for PyTorch 2.6 checkpoint loading.')

# Patch 2: Single-speaker dataset collate assertion in dataset.py
dataset_py = Path(f'{PIPER_SRC}/src/python/piper_train/vits/dataset.py')
if dataset_py.exists():
    ds_content = dataset_py.read_text()
    old_code = 'if utt.speaker_id is not None:'
    new_code = 'if self.is_multispeaker and (utt.speaker_id is not None):'
    if old_code in ds_content:
        ds_content = ds_content.replace(old_code, new_code)
        dataset_py.write_text(ds_content)
        print('✅ Patched piper_train/vits/dataset.py for single-speaker dataset collate.')

# Compile monotonic_align using cython + gcc
mono_dir = Path(f'{PIPER_SRC}/src/python/piper_train/vits/monotonic_align')
out_subdir = mono_dir / 'monotonic_align'
out_subdir.mkdir(parents=True, exist_ok=True)

r1 = subprocess.run([sys.executable, '-m', 'cython', '-3', 'core.pyx'],
                    cwd=mono_dir, capture_output=True, text=True)
print(r1.stderr if r1.returncode != 0 else '✅ Cython: core.pyx → core.c')

py_inc = sysconfig.get_path('include')
suffix = sysconfig.get_config_var('EXT_SUFFIX')
out_so = str(out_subdir / f'core{suffix}')
r2 = subprocess.run(
    ['gcc', '-shared', '-fPIC', '-O2', f'-I{py_inc}', 'core.c', '-o', out_so],
    cwd=mono_dir, capture_output=True, text=True
)
print(r2.stderr if r2.returncode != 0 else f'✅ GCC: core.c → {out_so}')

print('\n✅ piper_train installed & fully patched!')

---
## 📦 Section 2: Dataset Download & Preparation

- **2.1** Downloads HuggingFace dataset + base `.ckpt` + `config.json`.
- **2.2** Converts audio → 22050 Hz WAVs, writes `metadata.csv`, copies `config.json`, and runs Python preprocessor to phonemize text into `dataset.jsonl`.

In [ ]:
# 2.1 Download Dataset & Base Checkpoint
!python scripts/download_dataset.py --config configs/experiment001.yaml

In [ ]:
# 2.2 Prepare Dataset (wavs/ + metadata.csv + config.json + dataset.jsonl)
!python scripts/prepare_dataset.py --config configs/experiment001.yaml

In [ ]:
# 2.3 Verify Required piper_train Inputs
from pathlib import Path
processed_dir = Path('/content/drive/MyDrive/Arabic-Piper/processed/experiment001')
required = ['config.json', 'dataset.jsonl', 'metadata.csv', 'wavs']
all_ok = True
for name in required:
    exists = (processed_dir / name).exists()
    print(f'  {"✅" if exists else "❌ MISSING"}  {name}')
    if not exists: all_ok = False
if all_ok:
    n_wav  = len(list((processed_dir / 'wavs').glob('*.wav')))
    n_jl   = sum(1 for _ in open(processed_dir / 'dataset.jsonl'))
    print(f'\n✅ Ready — {n_wav} WAVs, {n_jl} phonemized entries.')
else:
    print('\n❌ Missing files — re-run Cells 2.1 and 2.2.')

---
## 🔊 Section 3: Baseline Benchmark

In [ ]:
# 3.1 Download Base ONNX Model
!mkdir -p /content/drive/MyDrive/Arabic-Piper/checkpoints/base/
!wget -q -O /content/drive/MyDrive/Arabic-Piper/checkpoints/base/ar_JO-kareem-medium.onnx \
    https://huggingface.co/rhasspy/piper-voices/resolve/v1.0.0/ar/ar_JO/kareem/medium/ar_JO-kareem-medium.onnx
!wget -q -O /content/drive/MyDrive/Arabic-Piper/checkpoints/base/ar_JO-kareem-medium.onnx.json \
    https://huggingface.co/rhasspy/piper-voices/resolve/v1.0.0/ar/ar_JO/kareem/medium/ar_JO-kareem-medium.onnx.json
print('✅ ONNX model downloaded.')

In [ ]:
# 3.2 Run Baseline Benchmark
!python scripts/benchmark.py \
    --model /content/drive/MyDrive/Arabic-Piper/checkpoints/base/ar_JO-kareem-medium.onnx \
    --model-config /content/drive/MyDrive/Arabic-Piper/checkpoints/base/ar_JO-kareem-medium.onnx.json \
    --sentences benchmark/benchmark_sentences.txt \
    --output-dir /content/drive/MyDrive/Arabic-Piper/outputs/baseline_benchmark

In [ ]:
# 3.3 Show Results
import json
from IPython.display import Audio, display, HTML
from pathlib import Path
report = Path('/content/drive/MyDrive/Arabic-Piper/outputs/baseline_benchmark/benchmark_report.json')
if report.exists():
    d = json.loads(report.read_text())
    print(f"Avg RTF: {d.get('avg_rtf')} | Duration: {d.get('total_audio_duration_sec')}s")
    wav = Path('/content/drive/MyDrive/Arabic-Piper/outputs/baseline_benchmark/benchmark_01.wav')
    if wav.exists():
        display(HTML('<h4>🔊 Baseline Sample:</h4>'))
        display(Audio(str(wav)))

---
## 🏋️ Section 4: Fine-Tuning

TensorBoard logs appear in the `lightning_logs/` sub-folder once training begins. Refresh it after Cell 4.3 starts.

In [ ]:
# 4.1 Checkpoint Detection & Target Epoch Calculation
import re
from pathlib import Path

ckpt_dir = Path('/content/drive/MyDrive/Arabic-Piper/checkpoints/experiment001')
ckpt_dir.mkdir(parents=True, exist_ok=True)
existing = sorted(ckpt_dir.glob('*.ckpt'))

BASE_EPOCH = 5079      # Base Kareem checkpoint epoch
FINE_TUNE_EPOCHS = 50  # Additional fine-tuning epochs to train
TARGET_MAX_EPOCHS = BASE_EPOCH + FINE_TUNE_EPOCHS  # 5129 total epochs

if existing:
    resume_ckpt = str(existing[-1])
    print(f'✅ Resuming from fine-tuning checkpoint: {resume_ckpt}')
else:
    base = Path('/content/drive/MyDrive/Arabic-Piper/checkpoints/base/ar/ar_JO/kareem/medium/epoch=5079-step=1682020.ckpt')
    resume_ckpt = str(base) if base.exists() else ''
    print(f'🆕 Fine-tuning from base checkpoint (epoch {BASE_EPOCH} → {TARGET_MAX_EPOCHS}): {resume_ckpt}')

# Export variables for next cell
os.environ['RESUME_CKPT'] = resume_ckpt
os.environ['TARGET_MAX_EPOCHS'] = str(TARGET_MAX_EPOCHS)

In [ ]:
# 4.2 Launch TensorBoard
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/Arabic-Piper/checkpoints/experiment001/lightning_logs

In [ ]:
# 4.3 Execute Training
# Note: --max_epochs specifies the total target epoch count (5079 base + 50 fine-tune = 5129)
!python -m piper_train \
    --dataset-dir /content/drive/MyDrive/Arabic-Piper/processed/experiment001 \
    --accelerator gpu \
    --devices 1 \
    --batch-size 16 \
    --validation-split 0.05 \
    --max_epochs $TARGET_MAX_EPOCHS \
    --checkpoint-epochs 5 \
    --default_root_dir /content/drive/MyDrive/Arabic-Piper/checkpoints/experiment001 \
    --resume_from_checkpoint "$RESUME_CKPT"

---
## 📊 Section 5: Export & Comparison

In [ ]:
# 5.1 Export Best Checkpoint to ONNX
from pathlib import Path
ckpts = sorted(Path('/content/drive/MyDrive/Arabic-Piper/checkpoints/experiment001').glob('*.ckpt'))
if ckpts:
    ckpt = str(ckpts[-1])
    onnx = '/content/drive/MyDrive/Arabic-Piper/outputs/experiment001/ar_JO_finetuned.onnx'
    print(f'Exporting checkpoint: {ckpt}')
    !python scripts/export_model.py --checkpoint "{ckpt}" --output-onnx "{onnx}"
else:
    print('❌ No checkpoint found. Ensure training finished successfully.')

In [ ]:
# 5.2 Benchmark Fine-Tuned ONNX Model
from pathlib import Path
model_path = Path('/content/drive/MyDrive/Arabic-Piper/outputs/experiment001/ar_JO_finetuned.onnx')
config_path = Path('/content/drive/MyDrive/Arabic-Piper/outputs/experiment001/ar_JO_finetuned.onnx.json')
sentences_path = Path('benchmark/benchmark_sentences.txt')
out_dir = Path('/content/drive/MyDrive/Arabic-Piper/outputs/finetuned_benchmark')

if model_path.exists():
    !python scripts/benchmark.py \
        --model "{model_path}" \
        --model-config "{config_path}" \
        --sentences "{sentences_path}" \
        --output-dir "{out_dir}"
else:
    print(f'❌ Fine-tuned model not found at {model_path}. Run Cell 5.1 first.')

In [ ]:
# 5.3 Side-by-Side Audio & Metric Comparison
import json
from pathlib import Path
from IPython.display import Audio, display, HTML

base_dir = Path('/content/drive/MyDrive/Arabic-Piper/outputs/baseline_benchmark')
ft_dir   = Path('/content/drive/MyDrive/Arabic-Piper/outputs/finetuned_benchmark')

base_report = base_dir / 'benchmark_report.json'
ft_report   = ft_dir / 'benchmark_report.json'

if base_report.exists() and ft_report.exists():
    b_data = json.loads(base_report.read_text())
    f_data = json.loads(ft_report.read_text())
    
    print('='*60)
    print(f"📊 Benchmark Summary:")
    print(f"   Baseline Average RTF   : {b_data.get('avg_rtf')}")
    print(f"   Fine-Tuned Average RTF: {f_data.get('avg_rtf')}")
    print('='*60)
    
    b_details = {item['id']: item for item in b_data.get('details', [])}
    f_details = {item['id']: item for item in f_data.get('details', [])}
    
    for item_id in sorted(f_details.keys()):
        text = f_details[item_id]['text']
        b_wav = base_dir / f"benchmark_{item_id:02d}.wav"
        f_wav = ft_dir / f"benchmark_{item_id:02d}.wav"
        
        display(HTML(f'<h4>Sample {item_id}: <i>"{text}"</i></h4>'))
        if b_wav.exists():
            display(HTML('<b>🔊 Baseline:</b>'))
            display(Audio(str(b_wav)))
        if f_wav.exists():
            display(HTML('<b>🎙️ Fine-Tuned:</b>'))
            display(Audio(str(f_wav)))
else:
    print('❌ Missing benchmark reports. Ensure Cells 3.2 and 5.2 completed.')

---
## ✅ Done!

Fine-tuned model saved at:
```
/content/drive/MyDrive/Arabic-Piper/outputs/experiment001/ar_JO_finetuned.onnx
```
Test locally:
```bash
python scripts/test_local.py --mode finetuned --model ar_JO_finetuned.onnx --text "السَّلَامُ عَلَيْكُمْ"
```